# NOTEBOOK 03 — DATA CLEANING
## PostgreSQL → Missing → Interpolation → Duplicate → Outlier → Logic Validation → PostgreSQL

### Mục tiêu
- Đọc dữ liệu tích hợp từ PostgreSQL.
- Tạo chuỗi tháng liên tục cho 5 thành phố.
- Đánh giá interpolation bằng **contiguous block masking**, không chọn phương pháp theo cảm tính.
- Giữ riêng target quan sát thật và target đã nội suy.
- Kiểm tra duplicate theo business key.
- Phát hiện outlier theo `City + Month`, không xóa cực trị khí hậu một cách máy móc.
- So sánh Before/After.
- Lưu bảng `climate_clean`.

## I. Setup và đọc dữ liệu từ PostgreSQL

In [1140]:
from pathlib import Path
import os

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"
APP_DIR = PROJECT_ROOT / "app"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Global Climate Change


In [1141]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url_object = URL.create(
    "postgresql",
    username="postgres",
    password="123456",
    host="localhost",
    port=5432,
    database="climate_change_db1",
)
engine = create_engine(url_object, pool_pre_ping=True)

with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

Database: climate_change_db1


In [1142]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

df = pd.read_sql(
    text("""
        SELECT *
        FROM vw_top5_country_climate
        ORDER BY country, dt
    """),
    engine,
    parse_dates=["dt"]
)

required_columns = {
    "dt",
    "country",
    "average_temperature"
}

missing = required_columns - set(df.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

print("✅ Country-level schema OK")

print("Shape:", df.shape)
display(df.head())


✅ Country-level schema OK
Shape: (13427, 15)


,dt,country,average_temperature,average_temperature_uncertainty,global_temperature,global_uncertainty,city_country_avg_temperature,city_country_avg_uncertainty,city_records,state_country_avg_temperature,state_country_avg_uncertainty,state_records,major_city_avg_temperature,major_city_avg_uncertainty,major_city_records
0,1852-07-01,Australia,14.116,1.530,14.512,0.778,11.066250,1.371500,12,9.158333,1.538667,6.0,10.9035,1.2890,2.0
1,1852-08-01,Australia,15.330,1.400,13.304,0.766,11.035750,1.206667,12,9.808667,1.405333,6.0,10.8580,1.1675,2.0
2,1852-09-01,Australia,18.740,1.446,11.478,0.689,12.731833,1.338167,12,12.535833,1.418500,6.0,12.3985,1.2220,2.0
3,1852-10-01,Australia,21.984,1.493,8.910,0.654,14.246000,1.814667,12,15.018667,1.860833,6.0,13.7065,1.8375,2.0
4,1852-11-01,Australia,24.073,1.466,4.593,0.693,15.935167,1.717417,12,17.459667,1.668167,6.0,15.2455,1.7150,2.0


## II. Audit trước Cleaning

In [1143]:
BUSINESS_KEY = ["dt", "country"]


## III. Tạo monthly calendar liên tục

Mỗi thành phố được reindex theo `MS` từ mốc bắt đầu hợp lệ đến tháng cuối.  
Các cột tĩnh (`city`, `country`, tọa độ) được forward/backward fill.  
Các context numeric được nội suy theo thời gian để phục vụ EDA, nhưng không dùng làm target.

In [1144]:
STATIC_COLS = [
    "country"
]

CONTEXT_COLS = [
    "global_temperature", "global_uncertainty",
    "city_country_avg_temperature", "city_country_avg_uncertainty",
    "city_records",
    "state_country_avg_temperature", "state_country_avg_uncertainty",
    "state_records",
    "major_city_avg_temperature", "major_city_avg_uncertainty",
    "major_city_records"
]
def complete_monthly(group):
    g = group.sort_values("dt").set_index("dt")
    start = max(pd.Timestamp("1850-01-01"), g.index.min())
    end = g.index.max()
    idx = pd.date_range(start, end, freq="MS")
    out = g.reindex(idx)
    out.index.name = "dt"

    for col in STATIC_COLS:
        if col in out.columns:
            out[col] = out[col].ffill().bfill()

    for col in CONTEXT_COLS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
            out[col] = out[col].interpolate(
                method="time",
                limit_direction="both"
            )

    return out.reset_index()

continuous = (
    df.groupby("country", group_keys=False)
      .apply(complete_monthly)
      .reset_index(drop=True)
)

display(
    continuous.groupby("country")
              .agg(min_date=("dt", "min"),
                   max_date=("dt", "max"),
                   rows=("dt", "size"))
)


KeyError: 'country'

## IV. Bảo toàn target gốc

- `temperature_observed`: nhiệt độ đo thật, dùng làm ground truth.
- `temperature_filled`: chuỗi liên tục sau interpolation.
- `target_is_observed`: 1 nếu có quan sát thật.
- `temperature_was_imputed`: 1 nếu giá trị được nội suy.

Nhờ vậy model không bị đánh giá bằng nhãn do chính interpolation tạo ra.

In [ ]:
continuous["average_temperature_observed"] = continuous["average_temperature"]
continuous["target_is_observed"] = (
    continuous["average_temperature_observed"].notna().astype("int8")
)

continuous["average_temperature_uncertainty"] = pd.to_numeric(
    continuous["average_temperature_uncertainty"], errors="coerce"
)

continuous["average_temperature_uncertainty"] = (
    continuous.groupby("country")["average_temperature_uncertainty"]
              .transform(
                  lambda s: s.interpolate(
                      method="linear",
                      limit_direction="both"
                  )
              )
)

## V. Đánh giá Interpolation

Mô phỏng gap thật bằng cách che nhiều **block liên tiếp 6 tháng** ở nhiều thành phố.

Phương pháp thử:
- Linear
- Time
- Spline order 3
- Polynomial order 3

Chọn phương pháp có **mean RMSE thấp nhất** trên các block có ground truth đầy đủ.

In [ ]:
def interpolate_series(series, method):
    if method == "linear":
        return series.interpolate(method="linear", limit_direction="both")
    if method == "time":
        return series.interpolate(method="time", limit_direction="both")
    if method == "spline3":
        return series.interpolate(
            method="spline", order=3, limit_direction="both"
        )
    if method == "polynomial3":
        return series.interpolate(
            method="polynomial", order=3, limit_direction="both"
        )
    raise ValueError(method)

def score_contiguous_gaps(
    series,
    method,
    block_months=6,
    n_blocks=5,
    seed=42
):
    series = series.sort_index()
    candidates = []

    for start in range(12, len(series) - block_months - 12):
        idx = series.index[start:start + block_months]
        if series.loc[idx].notna().all():
            candidates.append(start)

    if not candidates:
        return []

    rng = np.random.default_rng(seed)
    starts = rng.choice(
        candidates,
        size=min(n_blocks, len(candidates)),
        replace=False
    )

    rows = []
    for start in starts:
        idx = series.index[start:start + block_months]
        y_true = series.loc[idx].copy()
        masked = series.copy()
        masked.loc[idx] = np.nan

        try:
            filled = interpolate_series(masked, method)
            y_pred = filled.loc[idx]
            if y_pred.isna().any():
                continue
            rows.append({
                "MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": mean_squared_error(y_true, y_pred) ** 0.5
            })
        except Exception:
            continue

    return rows

methods = ["linear", "time", "spline3", "polynomial3"]
score_rows = []

for country, group in continuous.groupby("country"):
    s = group.set_index("dt")["average_temperature_observed"]

    for method in methods:
        scores = score_contiguous_gaps(s, method)
        for block_no, metrics in enumerate(scores, start=1):
            score_rows.append({
                "country": country,
                "method": method,
                "block": block_no,
                **metrics
            })

interp_scores = pd.DataFrame(score_rows)

if interp_scores.empty:
    BEST_METHOD = "time"
    print("Không đủ block hợp lệ để benchmark. Fallback =", BEST_METHOD)
else:
    summary = (
        interp_scores.groupby("method")[["MAE", "RMSE"]]
                     .agg(["mean", "std", "count"])
    )
    display(summary)

    BEST_METHOD = (
        interp_scores.groupby("method")["RMSE"]
                     .mean()
                     .sort_values()
                     .index[0]
    )
    print("BEST_METHOD =", BEST_METHOD)

## VI. Áp dụng interpolation và tạo cờ imputation

In [ ]:
def fill_country_target(country_name, group):
    g = group.sort_values("dt").copy()
    g["country"] = country_name
    s = g.set_index("dt")["average_temperature_observed"]
    filled = interpolate_series(s, BEST_METHOD)

    g["average_temperature_filled"] = filled.to_numpy()
    g["temperature_was_imputed"] = (
        g["average_temperature_observed"].isna()
        & g["average_temperature_filled"].notna()
    ).astype("int8")
    
    if "target_is_observed" not in g.columns:
        g["target_is_observed"] = g["average_temperature_observed"].notna().astype("int8")

    return g

clean_groups = []
for country_name, group in continuous.groupby("country", sort=False):
    clean_groups.append(fill_country_target(country_name, group))

clean = pd.concat(clean_groups, ignore_index=True)

for col in ["country", "dt", "average_temperature_observed", "average_temperature_filled", "target_is_observed", "temperature_was_imputed"]:
    assert col in clean.columns, f"Missing {col}"

display(
    clean.groupby("country")[
        ["target_is_observed", "temperature_was_imputed"]
    ].sum()
)


## VII. Duplicate

Khóa logic: `dt + city + country`.  
Không dùng `DataFrame.duplicated()` trên toàn bộ cột vì `dt` có thể nằm ngoài columns ở các bước khác.

In [ ]:
dup_count = int(clean.duplicated(BUSINESS_KEY).sum())
print("Duplicate business key:", dup_count)

if dup_count:
    display(
        clean[clean.duplicated(BUSINESS_KEY, keep=False)]
        .sort_values(BUSINESS_KEY)
        .head(30)
    )
    clean = clean.drop_duplicates(BUSINESS_KEY, keep="first").copy()

## VIII. Outlier Detection theo City + Month

Nhiệt độ cực trị có thể là dữ liệu khí hậu thật.  
Do đó:

- dùng robust z-score dựa trên median và MAD;
- so trong cùng `city + month`;
- chỉ **flag**, không tự động xóa.

In [ ]:
clean["month"] = clean["dt"].dt.month

seasonal_stats = (
    clean.groupby(["country", "month"])["average_temperature_filled"]
         .agg(
             seasonal_median="median",
             seasonal_mad=lambda s: np.median(
                 np.abs(s - np.median(s))
             )
         )
         .reset_index()
)

clean = clean.merge(
    seasonal_stats,
    on=["country", "month"],
    how="left"
)

mad = clean["seasonal_mad"].replace(0, np.nan)

clean["robust_z"] = (
    0.6745
    * (clean["average_temperature_filled"] - clean["seasonal_median"])
    / mad
)

clean["temperature_outlier_flag"] = (
    clean["robust_z"].abs().gt(4.5).fillna(False).astype("int8")
)

print("Outlier flags:", int(clean["temperature_outlier_flag"].sum()))

## IX. Logic Validation

In [ ]:
clean["invalid_temperature_flag"] = (
    (clean["average_temperature_filled"] < -90)
    | (clean["average_temperature_filled"] > 60)
).astype("int8")

clean["invalid_uncertainty_flag"] = (
    clean["average_temperature_uncertainty"] < 0
).fillna(False).astype("int8")

logic_cols = [
    "invalid_temperature_flag",
    "invalid_uncertainty_flag"
]
display(clean[logic_cols].sum().to_frame("count"))



## X. Before / After

In [ ]:
after = {
    "rows": len(clean),
    "columns": clean.shape[1],
    "missing_cells": int(clean.isna().sum().sum()),
    "duplicate_business_key": int(clean.duplicated(BUSINESS_KEY).sum()),
    "missing_observed_target": int(clean["average_temperature_observed"].isna().sum()),
    "missing_filled_target": int(clean["average_temperature_filled"].isna().sum()),
    "outlier_flags": int(clean["temperature_outlier_flag"].sum()),
    "invalid_values": int(clean[logic_cols].sum().sum())
}

comparison = pd.DataFrame({
    "Before": pd.Series(before),
    "After": pd.Series(after)
})
display(comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for country, group in clean.groupby("country"):
    g = group[group["dt"] >= "1980-01-01"]
    ax.plot(
        g["dt"],
        g["average_temperature_filled"],
        label=country,
        alpha=0.75
    )

ax.set_title("Cleaned monthly temperature — 1980 onward")
ax.set_ylabel("°C")
ax.legend(ncol=5)
plt.show()

## XI. Lưu `climate_clean` vào PostgreSQL

Loại các cột trung gian median/MAD, giữ:
- target observed,
- target filled,
- imputation flag,
- outlier/logic flags,
- context từ 4 bảng phụ.

In [ ]:
drop_cols = ["seasonal_median", "seasonal_mad"]
clean_to_save = clean.drop(
    columns=[c for c in drop_cols if c in clean.columns]
)

clean_to_save.to_sql(
    "climate_country_clean",
    engine,
    if_exists="replace",
    index=False,
    chunksize=20_000,
    method="multi"
)

with engine.begin() as conn:
    conn.execute(text(
        "CREATE INDEX IF NOT EXISTS idx_climate_country_clean_country_dt "
        "ON climate_country_clean(country, dt)"
    ))

print("Saved climate_country_clean:", clean_to_save.shape)

## XII. Kết luận Notebook 03

Phải báo cáo sau khi chạy:
- phương pháp interpolation thắng;
- MAE/RMSE interpolation;
- số điểm được nội suy;
- duplicate bị loại;
- số outlier được flag;
- invalid values;
- Before/After.

**Ground truth modeling ở Notebook 06 chỉ dùng `temperature_observed`, không dùng nhãn nội suy.**